# SLM-WM 单 Prompt GPU 方法资格化入口

该 Notebook 是正式批量实验之前的单 Prompt GPU 方法资格化入口。它只挂载 Google Drive、检出精确 clean detached Git 提交、传入一个受治理 Prompt 身份并以 `python -I` 调用统一宿主 launcher。依赖准备、真实主方法运行、716维 JVP/VJP、PSD-CG、latent 写回、Q/K 归因、图像保持性、密钥归因和 PRG 核验均由 repository 脚本实现。

资格化结果固定不支持论文结论。无论方法门禁通过还是失败, Notebook 都会先把本次本地诊断目录完整复制到 Google Drive 的 `MyDrive/SLM/gpu_method_qualification/` 下, 逐文件复验 SHA-256, 再根据宿主退出码结束。

## 运行顺序

1. 在 Colab 中选择 GPU runtime, 建议使用显存不低于 24GB 的 L4、A100 或同等级设备。
2. 在 Colab 密钥中配置 `HF_TOKEN`, 且账号已获得冻结 SD3.5 模型的访问权限。
3. 把下方 `SLM_WM_REPOSITORY_COMMIT` 填为已推送到远端的精确40位小写 Git SHA。
4. 默认使用 `probe_paper` 的第一个受治理 Prompt。资格化不形成 `probe_paper` 统计结论。
5. 从上到下运行全部单元格。宿主非零退出时, 诊断文件仍会先复制并复验到 Drive, 然后 Notebook 明确失败。
6. Drive 最终目录结构为 `SLM/gpu_method_qualification/<论文层级>/<提交>/<Prompt ID>/<UTC 会话>/`。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"
SLM_WM_PROMPT_ID = "prompt_e7420a434ce33de6"
SLM_WM_REPOSITORY_COMMIT = ""

import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if SLM_WM_REPOSITORY_COMMIT:
    os.environ["SLM_WM_REPOSITORY_COMMIT"] = SLM_WM_REPOSITORY_COMMIT
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME

# Drive 只保存模型缓存和完成复制的资格化诊断, Git 与科学执行仍位于本地 /content。
drive_root = Path("/content/drive/MyDrive/SLM")
hf_home = drive_root / "model_cache" / "huggingface"
torch_home = drive_root / "model_cache" / "torch"
hf_home.mkdir(parents=True, exist_ok=True)
torch_home.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(hf_home))
os.environ.setdefault("TORCH_HOME", str(torch_home))

print({
    "paper_run_name": SLM_WM_PAPER_RUN_NAME,
    "prompt_id": SLM_WM_PROMPT_ID,
    "drive_root": str(drive_root),
    "drive_free_gib": round(shutil.disk_usage(drive_root).free / 1024**3, 2),
})

In [ ]:
import os
import re
import subprocess
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("运行前必须填写已推送到远端的精确40位小写 Git SHA")

workspace_dir = Path(
    os.environ.get(
        "SLM_WM_WORKSPACE_DIR",
        f"/content/slm_wm_repository_{repository_commit[:12]}",
    )
)
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
print({"workspace_dir": str(workspace_dir), "repository_commit": repository_commit})

In [ ]:
FORMAL_HOST_LAUNCHER = "scripts/run_formal_workflow_host.py"
print({"formal_host_launcher": FORMAL_HOST_LAUNCHER})

In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get("HF_TOKEN")
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = token_from_secret
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("必须在 Colab 密钥中配置可访问冻结模型的 HF_TOKEN")
print({"hf_token_available_for_scientific_child": True})

In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("当前 Colab runtime 缺少 nvidia-smi, 无法执行 CUDA 科学工作流")
device_query = subprocess.run(
    [
        nvidia_smi,
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=False,
    capture_output=True,
    text=True,
)
if device_query.returncode != 0 or not device_query.stdout.strip():
    raise RuntimeError("nvidia-smi 查询失败: " + device_query.stderr.strip())
print(device_query.stdout.strip())

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

qualification_session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
local_session_root = (
    Path("outputs/gpu_method_qualification")
    / SLM_WM_PAPER_RUN_NAME
    / repository_commit
    / SLM_WM_PROMPT_ID
    / qualification_session_id
)
scientific_output_root = local_session_root / "scientific_evidence"
result_path = local_session_root / "workflow_result.json"
stdout_path = local_session_root / "host_stdout.log"
stderr_path = local_session_root / "host_stderr.log"
local_session_root.mkdir(parents=True, exist_ok=False)

host_command = [
    sys.executable,
    "-I",
    FORMAL_HOST_LAUNCHER,
    "--root",
    ".",
    "--repository-commit",
    repository_commit,
    "qualification",
    "--paper-run-name",
    SLM_WM_PAPER_RUN_NAME,
    "--prompt-id",
    SLM_WM_PROMPT_ID,
    "--qualification-output-root",
    scientific_output_root.as_posix(),
    "--result-path",
    result_path.as_posix(),
]
with stdout_path.open("w", encoding="utf-8") as stdout_file, stderr_path.open(
    "w", encoding="utf-8"
) as stderr_file:
    host_process = subprocess.run(
        host_command,
        check=False,
        stdout=stdout_file,
        stderr=stderr_file,
        text=True,
    )

print(stdout_path.read_text(encoding="utf-8")[-8000:])
error_tail = stderr_path.read_text(encoding="utf-8")[-8000:]
if error_tail:
    print(error_tail, file=sys.stderr)

drive_session_parent = (
    drive_root
    / "gpu_method_qualification"
    / SLM_WM_PAPER_RUN_NAME
    / repository_commit
    / SLM_WM_PROMPT_ID
)
drive_session_dir = drive_session_parent / qualification_session_id
drive_staging_dir = drive_session_parent / f".{qualification_session_id}.copying"
drive_session_parent.mkdir(parents=True, exist_ok=True)
if drive_session_dir.exists() or drive_staging_dir.exists():
    raise RuntimeError("本次资格化 Drive 会话目录已经存在, 禁止覆盖既有诊断")
shutil.copytree(local_session_root, drive_staging_dir)

local_file_digests = {
    path.relative_to(local_session_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(local_session_root.rglob("*"))
    if path.is_file()
}
drive_file_digests = {
    path.relative_to(drive_staging_dir).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted(drive_staging_dir.rglob("*"))
    if path.is_file()
}
if not local_file_digests or drive_file_digests != local_file_digests:
    raise RuntimeError("资格化结果复制到 Google Drive 后未通过逐文件 SHA-256 复验")
drive_staging_dir.rename(drive_session_dir)
print({
    "host_return_code": host_process.returncode,
    "drive_session_dir": drive_session_dir.as_posix(),
    "persisted_file_count": len(drive_file_digests),
})

formal_workflow_result = (
    json.loads(result_path.read_text(encoding="utf-8"))
    if result_path.is_file()
    else {}
)
if host_process.returncode != 0:
    raise RuntimeError(
        "单 Prompt GPU 方法资格化未通过; 诊断已经落盘到 "
        + drive_session_dir.as_posix()
    )
if formal_workflow_result.get("decision") != "pass":
    raise RuntimeError("宿主返回0, 但资格化结果没有形成 pass 决策")
if formal_workflow_result.get("supports_paper_claim") is not False:
    raise RuntimeError("单 Prompt 资格化不得支持论文结论")
formal_workflow_result

In [ ]:
print(json.dumps(formal_workflow_result, ensure_ascii=False, sort_keys=True, indent=2))
print({"google_drive_result_dir": drive_session_dir.as_posix()})